# 2262 step 1: modelling dataset, baselines and benchmark gate

Builds the modelling dataset for the employment status model comparison, scores the two rolling-average
baselines, and checks the 2260 benchmark is reproduced before any model is tried.

1. **Features:** size band, dormancy level (with a "not recorded" level), as-of CQC rating, time registered, 5 location folds.
2. **Baselines:** the pipeline's rolling average as it is (**as-is**, the like-for-like comparison, not fold-safe) and a
   rolling average that leaves each fold's workplaces out (**fold-safe**, reference only).
3. **Two tests:** held-out workplaces (decides) and hiding each workplace's last 12 months (guard).
4. **Gate:** the fold-safe baseline must reproduce 2260's 3.60pp and 1.27pp.

**Before running:** the pipeline has run in the branch bucket, `sagemaker_setup.ipynb` has put the repo on the instance
(branch `main`) and the parameters below are set. Merged repo modules are imported from `REPO_DIR`; functions that are only on
the unmerged 2260 and 2103 branches are copied into the cells below.

**Writes** only to `RESULTS_PREFIX`: the modelling dataset, the hidden-months dataset and `results/baseline_scores.parquet`.

**Stop the notebook instance when you finish** - SageMaker bills while it is running.

## Parameters

- `REPO_DIR`, `DATA_BUCKET`, `RESULTS_PREFIX`, `STORAGE_OPTIONS`, `LOCAL` - where the repo, data and results live.
- `*_PATH` - inputs (imputed shares, filled posts estimates, CQC ratings) and outputs.
- `N_FOLDS`, `FOLD_SEED` - the 2260 location folds. `SAMPLE_PERCENT`, `SAMPLE_SEED` - the workplace sample used for jumpiness and the hidden-months test.
- `HIDDEN_MONTHS`, `ROLLING_AVERAGE_PERIOD`, `*_PERIOD` - the 2260 settings. `FY_START_MONTH` - headline years are financial years.
- Column names and feature levels are defined once here, including derived names.
- `EXPECTED_*`, `GATE_*` - the benchmark to reproduce. `GATE_LEVEL` is `"aggregate"` (service x region x role x date, as 2260's utils score it) or `"share_pooled"`.

In [ ]:
REPO_DIR = "/home/ec2-user/SageMaker/DataEngineering"  # clone made by sagemaker_setup.ipynb, on main
DATA_BUCKET = "sfc-2262-employment-datasets"
RESULTS_PREFIX = "s3://skillsforcare-nonprod/sagemaker-notebooks/empstat-model-compare/"
STORAGE_OPTIONS = None  # e.g. {"aws_region": "eu-west-2"} if Polars can't find the region
LOCAL = False  # True when RESULTS_PREFIX is a local folder

SHARES_PATH = f"s3://{DATA_BUCKET}/domain=03_ind_cqc/dataset=02_employment_status_03_impute/"
ESTIMATES_PATH = f"s3://{DATA_BUCKET}/domain=03_ind_cqc/dataset=01_filled_posts_05_estimated/"
RATINGS_PATH = f"s3://{DATA_BUCKET}/domain=02_sfc/dataset=sfc_cqc_ratings_for_data_requests/"
MODELLING_DATA_PATH = f"{RESULTS_PREFIX}modelling_dataset.parquet"
HIDDEN_TREND_PATH = f"{RESULTS_PREFIX}hidden_trendline.parquet"
HIDDEN_DATA_PATH = f"{RESULTS_PREFIX}hidden_dataset.parquet"
RESULTS_DIR = f"{RESULTS_PREFIX}results/"

N_FOLDS = 5
FOLD_SEED = 42
SAMPLE_PERCENT = 10  # workplaces used for jumpiness and the hidden-months test
SAMPLE_SEED = 42
HIDDEN_MONTHS = 12
ROLLING_AVERAGE_PERIOD = "6mo"
EXTRAPOLATION_PERIOD = "2y"
INTERPOLATION_CAP_PERIOD = "5y"
FY_START_MONTH = 4  # headline years are financial years starting this month

# Column names are defined once, here.
LOCATION = "locationId"
DATE = "cqc_location_import_date"
SERVICE = "primary_service_type"
REGION = "current_Region"
CSSR = "current_CSSR"
ROLE = "published_job_role_label"
WORKERS = "estimate_filled_posts_by_job_role"
POSTS = "estimate_filled_posts"
TIME_REGISTERED = "time_registered"  # months
DORMANCY = "dormancy"
TIME_SINCE_DORMANT = "time_since_dormant"
RATING_DATE = "rating_date"
OVERALL_RATING = "Overall_rating"
STATUSES = ["permanent", "temporary", "bank_or_pool", "agency", "other"]
KNOWN = [f"ascwds_emp_stat_{status}_pct" for status in STATUSES]
TREND = [f"{column}_imputed_trendline" for column in KNOWN]
ROLLING = [f"{column}_rolling_avg" for column in KNOWN]
IMPUTED = [f"{column}_imputed" for column in KNOWN]
TRUTH = [f"{column}_truth" for column in KNOWN]
FOLD_SAFE = [f"{column}_fold_safe" for column in ROLLING]
HIDDEN = "hidden"
FOLD = "fold"
CELL_KEYS = [SERVICE, REGION, ROLE]

# Features built here.
SIZE_BAND = "size_band"
DORMANCY_LEVEL = "dormancy_level"
RATING = "cqc_rating"
DORMANCY_RECORDED = "dormancy_recorded"
SIZE_BAND_EDGES = [(10, "<10"), (50, "10-49"), (150, "50-149")]  # upper edges, exclusive
SIZE_BAND_TOP = "150+"
SIZE_BAND_UNKNOWN = "unknown"
SIZE_BANDS = [label for _, label in SIZE_BAND_EDGES] + [SIZE_BAND_TOP, SIZE_BAND_UNKNOWN]
DORMANT_VALUE = "Y"
DORMANCY_NOT_RECORDED = "not recorded"
DORMANCY_NONE_YET = "none yet"
DORMANT_NOW = "dormant now"
DORMANT_RECENT = "dormant 1-12m ago"
DORMANT_LONG = "dormant 13m+ ago"
DORMANT_RECENT_MONTHS = 12
DORMANCY_LEVELS = [DORMANCY_NOT_RECORDED, DORMANCY_NONE_YET, DORMANT_NOW, DORMANT_RECENT, DORMANT_LONG]
RATINGS_BEST_FIRST = ["Outstanding", "Good", "Requires improvement", "Inadequate"]
UNRATED = "unrated"
RATING_LEVELS = RATINGS_BEST_FIRST + [UNRATED]

# Result column and metric names.
VARIANT, TEST, LEVEL, GROUP, METRIC, VALUE = "variant", "test", "level", "group", "metric", "value"
HELD_OUT, GUARD = "held_out", "hidden_months"
ROWS, MAE_PP, BRIER, TOP1 = "number_of_rows", "mae_pp", "brier_score", "top_1_accuracy"
MAE_SKILL, BRIER_SKILL = "mae_skill_score", "brier_skill_score"
BASELINE_MAE, BASELINE_BRIER = "baseline_mae", "baseline_brier_score"
TREND_MAE_PP, MEAN_CHANGE = "trend_mae_pp", "mean_period_to_period_change"
CONTRIBUTORS, WINDOW_CONTRIBUTORS, YEAR = "contributing_workplaces", "window_contributing_workplaces", "financial_year"

EXPECTED_HELD_OUT_PP = 3.60
EXPECTED_HIDDEN_PP = 1.27
GATE_TOLERANCE_PP = 0.05
GATE_LEVEL = "aggregate"
STOP_ON_GATE_FAIL = True

## Setup

In [ ]:
import sys

import polars as pl

sys.path.insert(0, REPO_DIR)
import projects._03_independent_cqc._02_employment_status.fargate.utils.impute_utils as imputeUtils
from projects._03_independent_cqc.utils import model_evaluation_utils as evaluation

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)

assert imputeUtils.PCT_COLUMNS == KNOWN and imputeUtils.IMPUTED_TRENDLINE_PCT_COLUMNS == TREND
assert imputeUtils.ROLLING_AVG_PCT_COLUMNS == ROLLING and imputeUtils.IMPUTED_PCT_COLUMNS == IMPUTED

## Input and output helpers

`scan` and `sink` read and write the large datasets lazily. `write_result` and `read_result` handle small result tables.
`in_sample` picks a fixed random share of workplaces by hashing their ID.

In [ ]:
import io
from pathlib import Path


def scan(path: str) -> pl.LazyFrame:
    return pl.scan_parquet(path, storage_options=STORAGE_OPTIONS)


def sink(lf: pl.LazyFrame, path: str) -> None:
    lf.sink_parquet(path, storage_options=STORAGE_OPTIONS)


def write_result(df: pl.DataFrame, name: str) -> None:
    """Write a small result as parquet under RESULTS_DIR."""
    if LOCAL:
        path = Path(RESULTS_DIR) / name
        path.parent.mkdir(parents=True, exist_ok=True)
        df.write_parquet(path)
        return
    import boto3

    bucket, key = RESULTS_DIR.removeprefix("s3://").split("/", 1)
    buffer = io.BytesIO()
    df.write_parquet(buffer)
    boto3.client("s3").put_object(Bucket=bucket, Key=f"{key.rstrip('/')}/{name}", Body=buffer.getvalue())


def read_result(name: str) -> pl.DataFrame:
    """Read a result written by `write_result`."""
    if LOCAL:
        return pl.read_parquet(Path(RESULTS_DIR) / name)
    import boto3

    bucket, key = RESULTS_DIR.removeprefix("s3://").split("/", 1)
    body = boto3.client("s3").get_object(Bucket=bucket, Key=f"{key.rstrip('/')}/{name}")["Body"].read()
    return pl.read_parquet(io.BytesIO(body))


in_sample = pl.col(LOCATION).hash(SAMPLE_SEED) % 100 < SAMPLE_PERCENT

## Feature: size band

Size band comes from `estimate_filled_posts`, treated as known for every workplace (it never uses the workplace's own
employment status counts). Cut-points: under 10, 10-49, 50-149, 150 and over, and "unknown" without an estimate.
The check covers each boundary (9, 10, 49, 50, 149, 150) and a null.

In [ ]:
def add_size_band(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Band the workplace's estimated filled posts; "unknown" where there is no estimate."""
    band = pl.when(pl.col(POSTS).is_null()).then(pl.lit(SIZE_BAND_UNKNOWN))
    for upper_edge, label in SIZE_BAND_EDGES:
        band = band.when(pl.col(POSTS) < upper_edge).then(pl.lit(label))
    return lf.with_columns(band.otherwise(pl.lit(SIZE_BAND_TOP)).cast(pl.Enum(SIZE_BANDS)).alias(SIZE_BAND))


posts = [None, 9, 10, 49, 50, 149, 150]
expected = [SIZE_BAND_UNKNOWN, "<10", "10-49", "10-49", "50-149", "50-149", "150+"]
checked = add_size_band(pl.LazyFrame({POSTS: posts}, schema={POSTS: pl.Float64})).collect()
assert checked[SIZE_BAND].cast(pl.String).to_list() == expected, checked
print("size band check passed")

## Feature: dormancy level

A null `dormancy` is "not recorded" (rows before the field existed). Otherwise: "none yet" (never dormant so far), "dormant now",
"dormant 1-12m ago" or "dormant 13m+ ago". The pre/post transition is scored separately later by whether the level is "not recorded".

In [ ]:
def add_dormancy_level(lf: pl.LazyFrame) -> pl.LazyFrame:
    """
    Describe dormancy as one level. A null dormancy is "not recorded" (before the field existed).
    Otherwise a null time since dormant means no dormancy yet, and time since dormant of 1 is dormant now.
    """
    dormancy = pl.col(DORMANCY).cast(pl.String)
    months_ago = pl.col(TIME_SINCE_DORMANT) - 1
    level = (
        pl.when(dormancy.is_null()).then(pl.lit(DORMANCY_NOT_RECORDED))
        .when(pl.col(TIME_SINCE_DORMANT).is_null()).then(pl.lit(DORMANCY_NONE_YET))
        .when(dormancy == DORMANT_VALUE).then(pl.lit(DORMANT_NOW))
        .when(months_ago <= DORMANT_RECENT_MONTHS).then(pl.lit(DORMANT_RECENT))
        .otherwise(pl.lit(DORMANT_LONG))
    )
    return lf.with_columns(level.cast(pl.Enum(DORMANCY_LEVELS)).alias(DORMANCY_LEVEL))


rows = [(None, None), ("N", None), ("Y", 1), ("N", 2), ("N", 13), ("N", 14)]
expected = [DORMANCY_NOT_RECORDED, DORMANCY_NONE_YET, DORMANT_NOW, DORMANT_RECENT, DORMANT_RECENT, DORMANT_LONG]
frame = pl.LazyFrame(
    {DORMANCY: [d for d, _ in rows], TIME_SINCE_DORMANT: [t for _, t in rows]},
    schema={DORMANCY: pl.String, TIME_SINCE_DORMANT: pl.Int64},
)
checked = add_dormancy_level(frame).collect()
assert checked[DORMANCY_LEVEL].cast(pl.String).to_list() == expected, checked
print("dormancy level check passed")

## Feature: as-of CQC rating

The latest real overall rating given on or before each date. Blank ratings are ignored, same-day ratings keep the best, and no
rating yet is "unrated" once joined.

In [ ]:
def add_asof_rating(lf: pl.LazyFrame, ratings_lf: pl.LazyFrame) -> pl.LazyFrame:
    """
    Add the latest real overall rating given on or before each row's date, null if none.

    Ratings on the same date keep the best. Blank ratings (such as "Inspected but not rated") are ignored.
    """
    best_first = pl.col(OVERALL_RATING).replace_strict(
        {rating: i for i, rating in enumerate(RATINGS_BEST_FIRST)}, default=None
    )
    ratings_lf = (
        ratings_lf.filter(pl.col(OVERALL_RATING).is_in(RATINGS_BEST_FIRST))
        .select(
            pl.col(LOCATION).cast(pl.String),
            pl.col(RATING_DATE).cast(pl.String).str.slice(0, 10).str.to_date().alias(RATING_DATE),
            pl.col(OVERALL_RATING).cast(pl.String),
        )
        .group_by(LOCATION, RATING_DATE)
        .agg(pl.col(OVERALL_RATING).sort_by(best_first).first())
        .sort(RATING_DATE)
    )
    return (
        lf.sort(DATE)
        .join_asof(ratings_lf, left_on=DATE, right_on=RATING_DATE, by=LOCATION, strategy="backward")
        .rename({OVERALL_RATING: RATING})
        .drop(RATING_DATE)
    )


from datetime import date

dates = [date(2018, 12, 1), date(2019, 6, 1), date(2020, 6, 1), date(2021, 1, 1)]
frame = pl.LazyFrame({LOCATION: ["a"] * 4, DATE: dates})
ratings = pl.LazyFrame(
    {
        LOCATION: ["a", "a", "a", "a"],
        RATING_DATE: ["2019-01-01", "2020-06-01", "2020-06-01", "2020-09-01"],
        OVERALL_RATING: ["Good", "Inadequate", "Requires improvement", "Inspected but not rated"],
    }
)
checked = add_asof_rating(frame, ratings).sort(DATE).collect()
assert checked[RATING].to_list() == [None, "Good", "Requires improvement", "Requires improvement"], checked
print("as-of rating check passed")

## Build and write the modelling dataset

One row per workplace, role and date: shares at every stage, the role's workers (the weight), the features and the fold.
Estimates are unique per workplace and date, so the left join adds no rows. Written once to `MODELLING_DATA_PATH`; later cells and notebooks scan it.

In [ ]:
shares_lf = scan(SHARES_PATH).select(
    pl.col(LOCATION).cast(pl.String), ROLE, DATE, SERVICE, REGION, WORKERS, *KNOWN, *TREND, *ROLLING, *IMPUTED
)
estimates_lf = scan(ESTIMATES_PATH).select(
    pl.col(LOCATION).cast(pl.String), DATE, CSSR, TIME_REGISTERED, POSTS, DORMANCY, TIME_SINCE_DORMANT
)
ratings_lf = scan(RATINGS_PATH)

estimates_lf = add_asof_rating(estimates_lf, ratings_lf)
modelling_lf = shares_lf.join(estimates_lf, on=[LOCATION, DATE], how="left")
modelling_lf = add_dormancy_level(add_size_band(modelling_lf))
modelling_lf = modelling_lf.with_columns(pl.col(RATING).fill_null(UNRATED).cast(pl.Enum(RATING_LEVELS)))
modelling_lf = evaluation.assign_location_folds(modelling_lf, LOCATION, N_FOLDS, FOLD_SEED)
modelling_lf = modelling_lf.select(
    LOCATION, ROLE, DATE, SERVICE, REGION, CSSR, WORKERS, FOLD,
    TIME_REGISTERED, SIZE_BAND, DORMANCY_LEVEL, RATING,
    *KNOWN, *TREND, *ROLLING, *IMPUTED,
)
sink(modelling_lf, MODELLING_DATA_PATH)

## Summary of the modelling dataset

Rows, workplaces, rows with known shares and the share of rows at each feature level. The size-band and dormancy checks above
ran on synthetic values; this checks the real data has the levels you expect (for example "not recorded" only in early periods).

In [ ]:
modelling_lf = scan(MODELLING_DATA_PATH)
print(
    modelling_lf.select(
        pl.len().alias("rows"),
        pl.col(LOCATION).n_unique().alias("workplaces"),
        pl.col(KNOWN[0]).count().alias("known_rows"),
        pl.col(IMPUTED[0]).count().alias("imputed_rows"),
    ).collect()
)
for feature in [SIZE_BAND, DORMANCY_LEVEL, RATING]:
    print(modelling_lf.group_by(feature).agg(pl.len().alias("rows")).sort("rows", descending=True).collect())
print(
    modelling_lf.filter(pl.col(DORMANCY_LEVEL) == DORMANCY_NOT_RECORDED)
    .select(pl.col(DATE).min().alias("first_not_recorded"), pl.col(DATE).max().alias("last_not_recorded"))
    .collect()
)

## Rolling-average functions (copied from 2260)

Rolling averages depend only on the cell (service, region, role, date, fold), so they are built from small per-cell totals rather
than rows. `rolling_averages` repeats the pipeline's `add_rolling_avg_pcts`; leaving one fold out gives the fold-safe version.

In [ ]:
def build_cells(lf: pl.LazyFrame) -> pl.DataFrame:
    """Total the trendline shares per service, region, role, date and fold, with contributing workplaces."""
    return (
        lf.group_by(*CELL_KEYS, DATE, FOLD)
        .agg(
            *[pl.col(column).cast(pl.Float64).sum() for column in TREND],
            pl.col(TREND[0]).is_not_null().sum().alias(CONTRIBUTORS),
        )
        .collect(engine="streaming")
    )


def rolling_averages(cells_lf: pl.LazyFrame, period: str, hold_out_fold: int | None) -> pl.LazyFrame:
    """
    Rolling average of the trendline shares per service, region and role, from cell totals.

    Matches the pipeline's `add_rolling_avg_pcts`: each workplace counts once and a date with no
    contributors takes the nearest average. Cells in `hold_out_fold` add nothing to any total.
    """

    def kept(column: str) -> pl.Expr:
        expr = pl.col(column)
        return expr if hold_out_fold is None else expr.filter(pl.col(FOLD) != hold_out_fold)

    date_totals_lf = cells_lf.group_by(*CELL_KEYS, DATE).agg(
        *[kept(column).sum() for column in TREND], kept(CONTRIBUTORS).sum()
    )
    rolled_lf = (
        date_totals_lf.sort(*CELL_KEYS, DATE)
        .rolling(index_column=DATE, group_by=CELL_KEYS, period=period)
        .agg(*[pl.col(column).sum() for column in TREND], pl.col(CONTRIBUTORS).sum().alias(WINDOW_CONTRIBUTORS))
    )
    return (
        rolled_lf.with_columns(
            pl.when(pl.col(WINDOW_CONTRIBUTORS) > 0)
            .then(pl.col(total) / pl.col(WINDOW_CONTRIBUTORS))
            .clip(0, 1)
            .cast(pl.Float32)
            .alias(rolling)
            for total, rolling in zip(TREND, ROLLING)
        )
        .with_columns(pl.col(ROLLING).forward_fill().backward_fill().over(CELL_KEYS, order_by=DATE))
        .select(*CELL_KEYS, DATE, *ROLLING)
    )


def fold_safe_rolling_averages(cells_df: pl.DataFrame) -> pl.DataFrame:
    """Each fold's rolling averages, built from the other folds' workplaces only."""
    return pl.concat(
        [
            rolling_averages(cells_df.lazy(), ROLLING_AVERAGE_PERIOD, fold)
            .select(*CELL_KEYS, DATE, pl.lit(fold, dtype=pl.UInt8).alias(FOLD), *[pl.col(r).alias(f) for r, f in zip(ROLLING, FOLD_SAFE)])
            .collect(engine="streaming")
            for fold in range(N_FOLDS)
        ]
    )

## Check the cell-based average against the pipeline column

With no fold left out the cell-based average must equal the pipeline's rolling average column. The largest difference is
printed and must be tiny (Float32 rounding); if not, stop and investigate before trusting the fold-safe baseline.

In [ ]:
cells_df = build_cells(modelling_lf)
check_lf = (
    modelling_lf.filter(pl.col(KNOWN[0]).is_not_null())
    .select(*CELL_KEYS, DATE, *ROLLING)
    .unique(subset=[*CELL_KEYS, DATE])
    .join(rolling_averages(cells_df.lazy(), ROLLING_AVERAGE_PERIOD, None), on=[*CELL_KEYS, DATE], suffix="_cells")
)
largest_difference = check_lf.select(
    pl.max_horizontal((pl.col(column) - pl.col(f"{column}_cells")).abs().max() for column in ROLLING)
).collect().item()
print(f"largest difference from the pipeline's rolling average: {largest_difference}")
assert largest_difference < 1e-4

## Hidden-months functions (copied from 2260)

`hide_last_months` nulls each workplace's known shares in its last 12 months with known values (counted from that workplace's last
known date), keeping the originals in `<column>_truth`. `impute_hidden` runs the pipeline's imputation, which fills gaps along the
`*_rolling_avg` columns. Rows with no known values are never hidden.

In [ ]:
def hide_last_months(lf: pl.LazyFrame, months: int = HIDDEN_MONTHS) -> pl.LazyFrame:
    """
    Null each workplace's known percentages in its last `months` months with known values,
    keeping the originals in "<column>_truth" and flagging the hidden rows in "hidden".
    """
    last_known_date = pl.col(DATE).filter(pl.col(KNOWN[0]).is_not_null()).max().over(LOCATION)
    hidden = pl.col(DATE) > last_known_date.dt.offset_by(f"-{months}mo")
    return lf.with_columns(
        *[pl.col(column).alias(truth) for column, truth in zip(KNOWN, TRUTH)],
        hidden.fill_null(False).alias(HIDDEN),
    ).with_columns(pl.when(pl.col(HIDDEN)).then(None).otherwise(pl.col(column)).alias(column) for column in KNOWN)


def impute_hidden(df: pl.DataFrame) -> pl.DataFrame:
    """Run the pipeline's imputation, which fills gaps along the ROLLING columns."""
    return imputeUtils.add_imputed_pcts(df.lazy()).collect(engine="streaming")

## Scoring functions

`score_shares` is copied from 2260. The rest score one variant in long format (`variant`, `test`, `level`, `group`, `metric`, `value`):

- `share_*` - row-level error, Brier, top 1 and skill scores pooled, by size band and by whether dormancy was recorded (the pre/post transition).
- `headline` - **the decision metric**: mean absolute error in pp, worker-weighted within service x region x role x financial year, then averaged over those cells. `headline_cssr` uses CSSR instead of region; `headline_fold` is per fold.
- `aggregate` - the 2260 measure (service x region x role x date, each workplace counting equally), used for the benchmark gate.
- `trend` - mean absolute gap (pp) between the national mean predicted and known share per date.
- `jumpiness_*` - mean absolute change between dates in the predictions' own units. **Reported only, never pass/fail.**

In [ ]:
def calculate_mean_absolute_error(estimates: list[pl.Expr], known_columns: list[str]) -> pl.Expr:
    return pl.mean_horizontal((estimate - pl.col(known)).abs() for estimate, known in zip(estimates, known_columns))


def calculate_summed_squared_error(estimates: list[pl.Expr], known_columns: list[str]) -> pl.Expr:
    return pl.sum_horizontal((estimate - pl.col(known)) ** 2 for estimate, known in zip(estimates, known_columns))


def score_shares(
    lf: pl.LazyFrame, predicted_columns: list[str], known_columns: list[str], by_columns: list[str] | None = None
) -> pl.LazyFrame:
    """
    Score predicted against known shares row by row, every row counting equally.

    Only rows with every known and predicted share are scored. The skill scores compare each error with
    a baseline predicting the pooled mean known share on every row. A tied prediction counts as its first
    column for top 1.
    """
    by_columns = by_columns or []
    predicted = [pl.col(column) for column in predicted_columns]
    baseline = [pl.col(column).mean() for column in known_columns]
    scored_lf = lf.filter(pl.all_horizontal(pl.col(known_columns + predicted_columns).is_not_null()))
    row_scores_lf = scored_lf.select(
        *by_columns,
        calculate_mean_absolute_error(predicted, known_columns).alias(MAE_PP),
        calculate_summed_squared_error(predicted, known_columns).alias(BRIER),
        (pl.concat_list(predicted_columns).list.arg_max() == pl.concat_list(known_columns).list.arg_max())
        .cast(pl.Float64)
        .alias(TOP1),
        calculate_mean_absolute_error(baseline, known_columns).alias(BASELINE_MAE),
        calculate_summed_squared_error(baseline, known_columns).alias(BASELINE_BRIER),
    )
    mae = pl.col(MAE_PP).mean()
    brier = pl.col(BRIER).mean()
    scores = [
        pl.len().alias(ROWS),
        (mae * 100).alias(MAE_PP),
        brier,
        pl.col(TOP1).mean(),
        (1 - mae / pl.col(BASELINE_MAE).mean()).alias(MAE_SKILL),
        (1 - brier / pl.col(BASELINE_BRIER).mean()).alias(BRIER_SKILL),
    ]
    return row_scores_lf.group_by(by_columns).agg(scores) if by_columns else row_scores_lf.select(scores)


def long_scores(df: pl.DataFrame, level: str, group_column: str | None = None) -> pl.DataFrame:
    index = [group_column] if group_column else []
    group = (pl.col(group_column).cast(pl.String) if group_column else pl.lit("all")).alias(GROUP)
    return (
        df.with_columns(pl.all().exclude(index).cast(pl.Float64))
        .unpivot(index=index, variable_name=METRIC, value_name=VALUE)
        .select(pl.lit(level).alias(LEVEL), group, METRIC, VALUE)
    )


def complete_rows(predicted: list[str], known: list[str]) -> pl.Expr:
    return pl.all_horizontal(pl.col(predicted + known).is_not_null())


def headline_errors(
    df: pl.DataFrame, predicted: list[str], known: list[str], geography: str, by_fold: bool = False
) -> pl.DataFrame:
    """
    Mean absolute error (pp) of predicted against known shares, worker-weighted within each service,
    geography, role and financial year, then averaged over those cells.
    """
    weight = pl.col(WORKERS).cast(pl.Float64).fill_null(0)
    year = (pl.col(DATE).dt.year() - (pl.col(DATE).dt.month() < FY_START_MONTH).cast(pl.Int32)).alias(YEAR)
    keys = [SERVICE, geography, ROLE, year] + ([FOLD] if by_fold else [])
    cells_lf = (
        df.lazy()
        .filter(complete_rows(predicted, known) & (weight > 0))
        .group_by(keys)
        .agg((pl.col(column).cast(pl.Float64) * weight).sum() / weight.sum() for column in predicted + known)
    )
    error = pl.mean_horizontal((pl.col(p) - pl.col(k)).abs() for p, k in zip(predicted, known))
    scores = [pl.len().alias(ROWS), (error.mean() * 100).alias(MAE_PP)]
    if by_fold:
        return cells_lf.group_by(FOLD).agg(scores).collect()
    return cells_lf.select(scores).collect()


def aggregate_error(df: pl.DataFrame, predicted: list[str], known: list[str]) -> pl.DataFrame:
    """Mean absolute error (pp) of the mean predicted against the mean known share per service, region, role and date."""
    cells_lf = (
        df.lazy()
        .filter(complete_rows(predicted, known))
        .group_by(SERVICE, REGION, ROLE, DATE)
        .agg(pl.col(predicted).mean(), pl.col(known).mean())
    )
    error = pl.mean_horizontal((pl.col(p) - pl.col(k)).abs() for p, k in zip(predicted, known))
    return cells_lf.select(pl.len().alias(ROWS), (error.mean() * 100).alias(MAE_PP)).collect()


def national_trend(df: pl.DataFrame, predicted: list[str], known: list[str]) -> pl.DataFrame:
    """Mean predicted and known share per date and status, in long format."""
    dates_lf = df.lazy().filter(complete_rows(predicted, known)).group_by(DATE).agg(pl.col(predicted).mean(), pl.col(known).mean())
    return pl.concat(
        [
            dates_lf.select(DATE, pl.lit(status).alias("status"), pl.col(p).alias("predicted_mean"), pl.col(k).alias("known_mean"))
            for status, p, k in zip(STATUSES, predicted, known)
        ]
    ).collect()


def jumpiness_scores(jitter_df: pl.DataFrame, predicted: list[str]) -> pl.DataFrame:
    """Mean absolute change between dates (own units) per workplace and role, and per service, region and role."""
    aggregate_lf = jitter_df.lazy().group_by(SERVICE, REGION, ROLE, DATE).agg(pl.col(predicted).mean())
    results = []
    for level, lf, partition in [
        ("jumpiness_workplace", jitter_df.lazy(), [LOCATION, ROLE]),
        ("jumpiness_aggregate", aggregate_lf, CELL_KEYS),
    ]:
        change_df = evaluation.mean_period_to_period_change(lf, predicted, partition, DATE).collect()
        results.append(
            pl.DataFrame({LEVEL: [level], GROUP: ["all"], METRIC: [MEAN_CHANGE], VALUE: [change_df[MEAN_CHANGE].mean()]})
        )
    return pl.concat(results)


def score_variant(
    variant: str, test: str, df: pl.DataFrame, predicted: list[str], known: list[str], jitter_df: pl.DataFrame
) -> pl.DataFrame:
    """
    Score one variant in long format.

    `df` holds the rows scored (known rows, or the hidden rows); `jitter_df` holds every row of the
    sampled workplaces, for jumpiness.
    """
    df = df.with_columns(
        pl.when(pl.col(DORMANCY_LEVEL) == DORMANCY_NOT_RECORDED)
        .then(pl.lit("not recorded"))
        .otherwise(pl.lit("recorded"))
        .alias(DORMANCY_RECORDED)
    )
    results = []
    for level, by in [("share_pooled", None), ("share_size_band", SIZE_BAND), ("share_dormancy", DORMANCY_RECORDED)]:
        scores_df = score_shares(df.lazy(), predicted, known, [by] if by else None).collect()
        results.append(long_scores(scores_df, level, by))
    results.append(long_scores(headline_errors(df, predicted, known, REGION), "headline"))
    results.append(long_scores(headline_errors(df, predicted, known, CSSR), "headline_cssr"))
    results.append(long_scores(headline_errors(df, predicted, known, REGION, by_fold=True), "headline_fold", FOLD))
    results.append(long_scores(aggregate_error(df, predicted, known), "aggregate"))
    trend_df = national_trend(df, predicted, known)
    trend_error = (trend_df["predicted_mean"] - trend_df["known_mean"]).abs().mean() * 100
    results.append(pl.DataFrame({LEVEL: ["trend"], GROUP: ["all"], METRIC: [TREND_MAE_PP], VALUE: [trend_error]}))
    results.append(jumpiness_scores(jitter_df, predicted))
    return pl.concat(results).with_columns(pl.lit(variant).alias(VARIANT), pl.lit(test).alias(TEST))


def metric_value(scores_df: pl.DataFrame, variant: str, test: str, level: str, metric: str, group: str = "all") -> float:
    row = scores_df.filter(
        (pl.col(VARIANT) == variant) & (pl.col(TEST) == test) & (pl.col(LEVEL) == level)
        & (pl.col(METRIC) == metric) & (pl.col(GROUP) == group)
    )
    return row[VALUE].item()

## Scoring checks

A perfect prediction scores zero everywhere. A prediction 0.1 too high for permanent on workers 10 of 40 gives a weighted
permanent share 0.025 out, so the mean over 5 statuses is 0.005, or 0.5pp.

In [ ]:
check_df = pl.DataFrame(
    {
        LOCATION: ["a", "a", "b", "b"],
        SERVICE: ["s"] * 4, REGION: ["r"] * 4, CSSR: ["c"] * 4, ROLE: ["x"] * 4,
        DATE: [date(2020, 1, 1), date(2020, 2, 1)] * 2,
        WORKERS: [10.0, 10.0, 30.0, 30.0], FOLD: pl.Series([0, 0, 1, 1], dtype=pl.UInt8),
        SIZE_BAND: ["<10"] * 4, DORMANCY_LEVEL: [DORMANCY_NOT_RECORDED] * 4,
        **{column: [0.2] * 4 for column in KNOWN},
        **{column: [0.2] * 4 for column in ROLLING},
    },
    schema_overrides={DATE: pl.Date},
)
perfect = score_variant("perfect", HELD_OUT, check_df, ROLLING, KNOWN, check_df)
assert metric_value(perfect, "perfect", HELD_OUT, "headline", MAE_PP) == 0
assert metric_value(perfect, "perfect", HELD_OUT, "aggregate", MAE_PP) == 0
assert metric_value(perfect, "perfect", HELD_OUT, "jumpiness_workplace", MEAN_CHANGE) == 0

off_df = check_df.with_columns(pl.when(pl.col(WORKERS) == 10).then(0.3).otherwise(0.2).alias(ROLLING[0]))
off = score_variant("off", HELD_OUT, off_df, ROLLING, KNOWN, off_df)
# permanent is 0.1 too high on weight 10 of 40 workers: weighted share 0.225 vs 0.2, mean over 5 statuses
assert abs(metric_value(off, "off", HELD_OUT, "headline", MAE_PP) - 0.5) < 1e-6, off
print("scoring checks passed")

## Held-out test: load known rows

Loads every row with known shares (the scored rows) and every row of the sampled workplaces (for jumpiness), plus the cell totals.
Predictions never see the scored workplace's own fold in the fold-safe baseline; the as-is baseline does (the pipeline column
includes the workplace in its own average), which is why it is the like-for-like comparison for models that use it as a feature.

In [ ]:
ROW_COLUMNS = [LOCATION, SERVICE, REGION, CSSR, ROLE, DATE, WORKERS, FOLD, SIZE_BAND, DORMANCY_LEVEL]
known_df = (
    modelling_lf.filter(pl.all_horizontal(pl.col(KNOWN).is_not_null()))
    .select(*ROW_COLUMNS, *KNOWN, *ROLLING)
    .collect(engine="streaming")
)
sample_df = modelling_lf.filter(in_sample).select(*ROW_COLUMNS, *ROLLING).collect(engine="streaming")

fold_safe_df = fold_safe_rolling_averages(cells_df)
join_keys = [*CELL_KEYS, DATE, FOLD]
known_df = known_df.join(fold_safe_df, on=join_keys, how="left")
sample_df = sample_df.join(fold_safe_df, on=join_keys, how="left")
print(f"known rows: {known_df.height:,}, sample rows: {sample_df.height:,}")

## Score both baselines on held-out workplaces

`as_is` is the pipeline's rolling average column. `fold_safe` is the reference that reproduces 2260. Each is scored on known
shares only.

In [ ]:
scores_dfs = [
    score_variant("as_is", HELD_OUT, known_df, ROLLING, KNOWN, sample_df),
    score_variant("fold_safe", HELD_OUT, known_df, FOLD_SAFE, KNOWN, sample_df),
]
held_out_scores_df = pl.concat(scores_dfs)
print(held_out_scores_df.filter(pl.col(LEVEL).is_in(["headline", "aggregate", "share_pooled", "trend"]) & pl.col(METRIC).is_in([MAE_PP, TREND_MAE_PP])))

## Hidden-months test: build the hidden data

Hides each workplace's last 12 months, re-runs the pipeline's trendline imputation on what is left, builds the rolling average from
the hidden data (every workplace hidden at once, so near the end of the data averages rest on less than in production) and writes
the result. Workplace-level features (rating, time registered) are not re-dated to the cut-off; this test does not handle that.

In [ ]:
hidden_lf = hide_last_months(
    modelling_lf.select(LOCATION, SERVICE, REGION, CSSR, ROLE, DATE, WORKERS, FOLD, TIME_REGISTERED, SIZE_BAND, DORMANCY_LEVEL, RATING, *KNOWN)
)
hidden_lf = imputeUtils.add_imputed_trendline_pcts(hidden_lf, EXTRAPOLATION_PERIOD, INTERPOLATION_CAP_PERIOD)
sink(hidden_lf, HIDDEN_TREND_PATH)

hidden_cells_df = build_cells(scan(HIDDEN_TREND_PATH))
hidden_rolling_lf = rolling_averages(hidden_cells_df.lazy(), ROLLING_AVERAGE_PERIOD, None)
sink(scan(HIDDEN_TREND_PATH).join(hidden_rolling_lf, on=[*CELL_KEYS, DATE], how="left"), HIDDEN_DATA_PATH)

## Score the baseline on the hidden months

Imputes the sampled workplaces along the hidden-data rolling average and scores the imputed shares on the hidden rows against
`<column>_truth`. With every workplace hidden, as-is and fold-safe are the same average, so one row (`as_is`) stands for both.

In [ ]:
guard_sample_df = scan(HIDDEN_DATA_PATH).filter(in_sample).collect(engine="streaming")
guard_imputed_df = impute_hidden(guard_sample_df)
guard_scores_df = score_variant(
    "as_is", GUARD, guard_imputed_df.filter(pl.col(HIDDEN)), IMPUTED, TRUTH, guard_imputed_df
)
print(guard_scores_df.filter(pl.col(LEVEL).is_in(["headline", "aggregate", "share_pooled"]) & (pl.col(METRIC) == MAE_PP)))

## Benchmark gate

The fold-safe baseline must reproduce 2260: **3.60pp** held-out and **1.27pp** hiding the last 12 months, within `GATE_TOLERANCE_PP`.
If either misses, nothing after this is trustworthy: fix the settings (fold seed, rolling window, level in `GATE_LEVEL`) before moving on.

In [ ]:
all_scores_df = pl.concat([held_out_scores_df, guard_scores_df])
held_out_pp = metric_value(all_scores_df, "fold_safe", HELD_OUT, GATE_LEVEL, MAE_PP)
guard_pp = metric_value(all_scores_df, "as_is", GUARD, GATE_LEVEL, MAE_PP)
held_out_ok = abs(held_out_pp - EXPECTED_HELD_OUT_PP) <= GATE_TOLERANCE_PP
guard_ok = abs(guard_pp - EXPECTED_HIDDEN_PP) <= GATE_TOLERANCE_PP
print(f"held-out: {held_out_pp:.3f}pp (expected {EXPECTED_HELD_OUT_PP}) {'PASS' if held_out_ok else 'FAIL'}")
print(f"hidden months: {guard_pp:.3f}pp (expected {EXPECTED_HIDDEN_PP}) {'PASS' if guard_ok else 'FAIL'}")
if STOP_ON_GATE_FAIL:
    assert held_out_ok and guard_ok, "Benchmark not reproduced: do not continue to the models"

## Results

Saves the long-format scores for the later notebooks, and shows the as-is baseline headline and guard numbers every model
has to beat by at least 1pp (and in all 5 folds).

In [ ]:
write_result(all_scores_df, "baseline_scores.parquet")
print(
    all_scores_df.filter(
        pl.col(LEVEL).is_in(["headline", "headline_cssr", "headline_fold", "trend"]) & pl.col(METRIC).is_in([MAE_PP, TREND_MAE_PP])
    ).sort(VARIANT, TEST, LEVEL, GROUP)
)